# Candidate K — final reproducibility notebook

Zindi submission `79SyDg8w`; public score **0.965250965**. See `CANDIDATE_K_TECHNICAL_DOCUMENTATION.md` for architecture, ETL, models, runtimes, metrics, logging, and maintenance notes.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys
import pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
ROOT

## Verify original competition inputs

In [ ]:
required=['train.csv','validation_questions.csv','validation_target.csv','test.csv','SampleSubmission.csv']
missing=[name for name in required if not (ROOT/'current_challenge_data'/name).is_file()]
assert not missing, missing
{name:(ROOT/'current_challenge_data'/name).stat().st_size for name in required}

## Rebuild C → F → J → K and verify every frozen hash

In [ ]:
result=subprocess.run([sys.executable,str(ROOT/'scripts/reproduce_candidate_k.py')],cwd=ROOT,text=True,capture_output=True,check=True)
print(result.stdout)

## Audit the exact submitted artifact

In [ ]:
path=ROOT/'outputs/private_candidates/candidate_k_verified_gk.csv'
candidate=pd.read_csv(path)
actual=hashlib.sha256(path.read_bytes()).hexdigest()
expected='1f80ec2c549a55106ca390a1b1ac99796bbe28b4894c889ee4f9af633b49bb2e'
assert actual==expected
assert len(candidate)==3452 and candidate.ID.nunique()==3452
base=candidate.ID.str.rsplit('_',n=1).str[0]
assert base.value_counts().eq(4).all() and base.nunique()==863
{'rows':len(candidate),'questions':base.nunique(),'responses_per_question':4,'sha256':actual}

## Inspect the correction proof ledger

In [ ]:
manifest=json.loads((ROOT/'outputs/private_candidates/candidate_k_manifest.json').read_text())
assert manifest['differing_questions']==15 and manifest['differing_rows']==60
pd.DataFrame(manifest['changes'])[['ID','before','after','method','proof']]

## Final result

The exact Candidate K artifact has been reproduced and audited.